<a href="https://colab.research.google.com/github/nayakamhrudaya/GenAI/blob/main/WebScraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================
# 1. INSTALL DEPENDENCIES
# =========================================
!pip install -q requests beautifulsoup4 openai

# =========================================
# 2. IMPORT LIBRARIES
# =========================================
import os
import requests
from bs4 import BeautifulSoup
from getpass import getpass
from openai import OpenAI

# =========================================
# 3. SET API KEY
# =========================================
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

client = OpenAI()

# =========================================
# 4. SCRAPER FUNCTION
# =========================================
def scrape_website(url):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)

        soup = BeautifulSoup(response.text, "html.parser")

        paragraphs = [p.get_text() for p in soup.find_all("p")]
        text = " ".join(paragraphs)

        return text[:5000]  # limit size
    except Exception as e:
        return f"Error: {e}"

# =========================================
# 5. SUMMARIZATION FUNCTION (LLM)
# =========================================
def summarize(text):
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": f"Summarize this webpage content clearly:\n{text}"}
            ],
            temperature=0
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# =========================================
# 6. EVALUATION FUNCTION
# =========================================
def evaluate(summary):
    if "Error" in summary:
        return "❌ Failed"
    elif len(summary) < 100:
        return "⚠️ Too short"
    else:
        return "✅ Good summary"

# =========================================
# 7. MAIN AGENT PIPELINE
# =========================================
def web_scraping_agent(url):
    print("🔍 Scraping website...")
    content = scrape_website(url)

    if content.startswith("Error"):
        return content

    print("🧠 Generating summary...")
    summary = summarize(content)

    print("📊 Evaluating output...")
    score = evaluate(summary)

    return summary, score

# =========================================
# 8. RUN THE AGENT
# =========================================
url = input("Enter a website URL: ")

result = web_scraping_agent(url)

print("\n==============================")
print("📄 RESULT")
print("==============================\n")

if isinstance(result, tuple):
    summary, score = result
    print("Summary:\n")
    print(summary)
    print("\nEvaluation:", score)
else:
    print(result)

# =========================================
# 9. MULTI-URL SUPPORT (OPTIONAL)
# =========================================
multi = input("\nRun multiple URLs? (y/n): ")

if multi.lower() == "y":
    urls = input("Enter URLs separated by comma:\n").split(",")

    for u in urls:
        print("\n-----------------------------------")
        print(f"Processing: {u.strip()}")
        res = web_scraping_agent(u.strip())

        if isinstance(res, tuple):
            print("Summary:", res[0][:300], "...")
            print("Evaluation:", res[1])
        else:
            print(res)

🔍 Scraping website...
🧠 Generating summary...
📊 Evaluating output...

📄 RESULT

Summary:

The webpage discusses the upcoming Pega Blueprint Case Study event scheduled for June 7-9, 2026, in Las Vegas, along with various offerings from Pega, including solutions, events, and resources for partners and customers. It highlights the role of AI agents—advanced software systems that utilize artificial intelligence to analyze, plan, and execute tasks autonomously. These agents can optimize business operations by continuously learning and adapting, thus improving efficiency and productivity.

AI agents can manage entire processes rather than just automating small tasks, and they operate by defining objectives and analyzing situations to take action through workflows. They are typically powered by large language models and require human oversight to ensure effective governance and adherence to business rules. The webpage emphasizes the importance of a centralized approach to logic and workflow a